# [Promote a model across environments](https://docs.databricks.com/aws/en/machine-learning/manage-model-lifecycle/#promote-a-model-across-environments)

(REF: https://github.com/databricks/mlops-stacks/blob/main/stack-customization.md) 

Databricks recommends that you deploy ML pipelines as code. This eliminates the need to promote models across environments, as all production models can be produced through automated training workflows in a production environment.

**However, there are cases where retraining models across various settings/environments may be cost-prohibitive or impractical. Instead, you can copy model versions across registered models in Unity Catalog to promote them across environments.** 

You need the following privileges to execute the example code below:

- **USE CATALOG** on the other (e.g. `staging` and `prod`) catalogs.
- **USE SCHEMA** on the `staging.ml_team and prod.ml_team` schemas.
- **EXECUTE** on {[env e.g. `prod`]_catalog_name}.{schema_name}.{model_name}.
- **In addition, you must be the owner of the registered model** e.g. {[env e.g. `prod`]_catalog_name}.{schema_name}.{model_name}.


- To promote a model, you typically use the copy_model_version MLflow Client API, enabling model versions to move from staging to production environments without needing to be in the same workspace: 
The following code snippet uses the copy_model_version MLflow Client API, available in MLflow version 2.8.0 and above.

```
%Python
import mlflow
mlflow.set_registry_uri("databricks-uc")

client = mlflow.tracking.MlflowClient()
src_model_name = "staging.ml_team.fraud_detection"
src_model_version = "1"
src_model_uri = f"models:/{src_model_name}/{src_model_version}"
dst_model_name = "prod.ml_team.fraud_detection"
copied_model_version = client.copy_model_version(src_model_uri, dst_model_name)
```

After the model version is in the production environment, you can perform any necessary pre-deployment validation. Then, you can mark the model version for deployment using aliases.

```
%Python
client = mlflow.tracking.MlflowClient()
client.set_registered_model_alias(name="prod.ml_team.fraud_detection", alias="Champion", version=copied_model_version.version)
```

In the example above, **only users who can read from the** staging.ml_team.fraud_detection **registered model and write to** the prod.ml_team.fraud_detection r**egistered model can promote staging models to the production environment.** The **_same users can also use aliases to manage which model versions are deployed within the production environment._** You don’t need to configure any other rules or policies to govern model promotion and deployment.

You can customize this flow to promote the model version across multiple environments that match your setup, such as `dev`, `qa`, and `prod`. Access control is enforced as configured in each environment.

### PERMISSIONS 

To promote model permissions as the owner of a registered model in Unity Catalog, it is indeed possible to use a service principal. In Unity Catalog, the access control is based on privileges assigned to users, groups, or service principals.

Permissions Required
To manage a registered model in Unity Catalog, you need the following:

USE SCHEMA and USE CATALOG privileges on the schema and catalog that contains the registered model.
CREATE_MODEL privilege on the schema to create new registered models.
When granting permissions, the service principal can be treated like any other user or group, allowing you to grant it the necessary privileges to manage the models effectively.

Granting Permissions Example
You can grant permissions to a service principal using the SQL GRANT command as follows:

GRANT CREATE_MODEL ON SCHEMA <schema-name> TO <principal-name>;
GRANT USE CATALOG ON CATALOG <catalog-name> TO <principal-name>;
SQL

This allows the specified service principal to create models and access the necessary catalog and schema1.

To set permissions programmatically, you can use the Grants REST API, ensuring that you specify the correct securable type for models, which is classified as a FUNCTION in Unity Catalog1.

### NOTES

- Databricks uses Unity Catalog to facilitate model management across different workspaces, allowing you to promote model versions from one registered model to another, even if they are in separate environments. This means models can be shared and promoted across different workspaces, provided they are connected to the same Unity Catalog metastore and appropriate privileges are in place.
- NB: While models can be promoted across different environments, those environments do not need to be located in the same workspace, as Unity Catalog provides the necessary governance and sharing capabilities.

- >`client.copy_model_version creates a deep copy by downloading model artifacts and reuploading them to a different storage location` 

## Some example/recommended pre-deployment validation steps:

Dependency Validation: 
- Ensure that all dependencies required by the model are correctly specified and can be installed in the target environment.
- Input Data Validation: Validate that the input data format and content are as expected by the model. 
- Performance Validation: Evaluate the model's performance on a validation dataset to ensure it meets the required performance criteria. 
- Compliance Checks: Ensure the model complies with any regulatory or organizational requirements. 
- Environment Validation: Test the model in an isolated environment to ensure it runs correctly with the specified dependencies and environment variables. 
- You can use the mlflow.models. predict API to perform these validations. Here is an example:


----     

```
%python
import mlflow

model_uri = "models:/prod.ml_team.fraud_detection/1"  # Use the model URI without run_id
input_data = {"col1": 34.2, "col2": 11.2, "col3": "green"}

mlflow.models.predict(
    model_uri=model_uri,
    input_data=input_data,
    content_type="json",
    env_manager="virtualenv",
    install_mlflow=False,
    pip_requirements_override=["pillow==10.3.0", "scipy==1.13.0"],
)
```

[The input_data is provided in JSON format, and the environment is managed using virtualenv with specific dependency overrides.]

In [0]:
%run ../_resources/00-setup $reset_all_data=false

In [0]:
# import mlflow
# mlflow.set_registry_uri("databricks-uc")

# client = mlflow.tracking.MlflowClient()
# src_model_name = "staging.ml_team.fraud_detection"
# src_model_version = "1"
# src_model_uri = f"models:/{src_model_name}/{src_model_version}"
# dst_model_name = "prod.ml_team.fraud_detection"
# copied_model_version = client.copy_model_version(src_model_uri, dst_model_name

In [0]:
import mlflow
mlflow_client = mlflow.MlflowClient()

# Make sure we use Mlflow with UC registry
mlflow.set_registry_uri('databricks-uc')

src_catalog = "mmt_demos"
src_db = "hls_readmission_dbdemoinit"

src_model_name = "dbdemos_hls_pr"
src_model_alias = "champion"  # update as appropriate: use champion/challenger

# Use the model URI with alias directly
src_model_uri = f"models:/{catalog}.{db}.{model_name}@{model_alias}"
print(src_model_uri)


In [0]:
input_dataset = {"dataframe_split": {"index": [0, 1, 2, 3, 4, 5, 6, 7, 8], "columns": ["MARITAL_M", "MARITAL_S", "RACE_asian", "RACE_black", "RACE_hawaiian", "RACE_other", "RACE_white", "ETHNICITY_hispanic", "ETHNICITY_nonhispanic", "GENDER_F", "GENDER_M", "INCOME", "BASE_ENCOUNTER_COST", "TOTAL_CLAIM_COST", "PAYER_COVERAGE", "enc_length", "ENCOUNTERCLASS_ambulatory", "ENCOUNTERCLASS_emergency", "ENCOUNTERCLASS_hospice", "ENCOUNTERCLASS_inpatient", "ENCOUNTERCLASS_outpatient", "ENCOUNTERCLASS_wellness", "age_at_encounter", "patient_id", "ENCOUNTER_ID"], "data": [[1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 892590, 85.55, 755.56, 755.56, 13500, 1, 0, 0, 0, 0, 0, 45.65366187542779, "e4ff91bc-bd28-3820-8e03-7966fdd0289f", "6730a65f-470c-a2c5-46bf-6c31bd3ab0dc"], [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 55722, 136.8, 777.55, 0.0, 900, 0, 0, 0, 0, 0, 1, 1.4373716632443532, "9f1c79c3-eb93-96db-292f-990b4084fda8", "c4827fef-7583-973f-5f6b-09f1b66555f2"], [0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 54368, 136.8, 979.11, 0.0, 900, 0, 0, 0, 0, 0, 1, 0.6899383983572895, "d44b76b7-88de-d9a0-d6dd-2395b8552739", "f3742f2c-da43-ceb0-2201-f561551033a3"], [0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 36123, 136.8, 791.94, 0.0, 2771, 0, 0, 0, 0, 0, 1, 50.116358658453116, "293ffee1-e1a5-1511-746e-2f55108d7985", "1d5728f3-b794-2307-3f2a-442f8e3e7442"], [0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 99024, 136.8, 1021.83, 817.46, 900, 0, 0, 0, 0, 0, 1, 3.9288158795345653, "35aaa13d-b480-a1b7-770e-0d02c8990341", "44a19292-e7ff-abed-f82b-b9211e0e1eae"], [1, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 169036, 142.58, 8646.42, 3149.49, 3004, 0, 0, 0, 0, 1, 0, 28.15605749486653, "ce3af08b-821d-5288-0e5a-88950f1265d1", "41d4cdc2-b215-e210-7066-bbc5d37ec648"], [1, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 28150, 136.8, 923.8, 0.0, 2378, 0, 0, 0, 0, 0, 1, 37.21834360027378, "12e088db-4ad6-5b3c-ae25-1b3103fd6585", "1a35fce9-6f4c-a40c-5daa-6209c90d5ed5"], [1, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 149217, 142.58, 11031.76, 11031.76, 900, 1, 0, 0, 0, 0, 0, 32.733744010951405, "83e4b8b0-1f23-fb55-294c-3b74159cd40e", "518a20c5-60fa-6732-8b24-342acc8bc77b"], [1, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 123273, 136.8, 9833.74, 7866.99, 3178, 0, 0, 0, 0, 0, 1, 25.182751540041068, "8e88c649-1f18-d322-9fd8-55e35cd014de", "57c8ef93-315c-3829-3016-35810c33df20"]]}}

In [0]:
src_model_uri

In [0]:
# import mlflow
# mlflow_client = mlflow.MlflowClient()

# # Define the destination model name
# dst_model_name = f"{catalog}{2}.{db}.{model_name}"

# # List all versions of the destination model
# model_versions = mlflow_client.search_model_versions(f"name='{dst_model_name}'")

# # Delete all versions of the destination model
# for version in model_versions:
#     mlflow_client.delete_model_version(name=dst_model_name, version=version.version)

# # Delete the registered model along with all its versions
# mlflow_client.delete_registered_model(name=dst_model_name)

# print(f"All versions of the model '{dst_model_name}' have been deleted.")

In [0]:
# src_model_uri = f"models:/{src_model_name}/{src_model_version}"

# mmt_demos{2}.hls_readmission_dbdemoinit
dst_model_name = f"{catalog}{2}.{db}.{model_name}" 
print(dst_model_name)

In [0]:
import mlflow
tracking_client = mlflow.tracking.MlflowClient() ## for copy_model_version
mlflow_client = mlflow.MlflowClient()

# Ensure the destination schema exists
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{db}")

# Copy the model version
copied_model_version = tracking_client.copy_model_version(src_model_uri, dst_model_name)

# Retrieve the version information of the copied model
copied_version_info = mlflow_client.get_model_version(name=dst_model_name, version=copied_model_version.version)

# Print the copied version information
print(copied_version_info)

In [0]:
tracking_client = mlflow.tracking.MlflowClient()
dst_model_alias = "champion"
tracking_client.set_registered_model_alias(name=dst_model_name, alias=dst_model_alias, version=copied_model_version.version)

In [0]:
dst_model_version = mlflow_client.get_model_version_by_alias(f"{catalog}{2}.{db}.{model_name}", model_alias).version  ##champion 

dst_model_uri = f"models:/{catalog}.{db}.{model_name}/{dst_model_version}"

print(dst_model_uri)

In [0]:
# {model_name}@{model_alias}|{model_latest_version}

In [0]:
mlflow.set_registry_uri('databricks-uc')

# Load model as a Spark UDF.
loaded_model = mlflow.pyfunc.spark_udf(
    spark, 
    model_uri=dst_model_uri, 
    result_type='double'
)
display(loaded_model)

In [0]:
features = loaded_model.metadata.get_input_schema().input_names()

#For this demo, reuse our dataset to test the batch inferences
test_dataset = spark.table('training_dataset')

patient_risk_df = (test_dataset 
                   .withColumn("risk_prediction", loaded_model(struct(*features)))          
                   .withColumn("model_info", 
                              #  F.lit(f"{catalog}.{db}.{model_name}@{model_alias}|{model_latest_version}")
                              F.lit(f"{dst_model_name}@{model_alias}|{dst_model_version}")
                              ) 
                   .withColumn("current_datetime", F.current_timestamp()) 
                   .withColumn("current_timestamp", F.to_unix_timestamp("current_datetime"))
                   .select(*features + ['ENCOUNTER_ID', 'PATIENT_ID', 'risk_prediction', 
                                        'model_info', 'current_datetime','current_timestamp'] ## added 
                          )
                  )

display(patient_risk_df)

In [0]:
## add any other validation metrics etc. 